# Knowledge Factor Investigation - Logistic

In [1]:
%pwd

'/Users/lichao/Library/CloudStorage/OneDrive-Personal/MLD01_Article/MLD01e_Code'

In [2]:
%cd ..

/Users/lichao/Library/CloudStorage/OneDrive-Personal/MLD01_Article


/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## Import PAckage

In [33]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report 

from xgboost import XGBClassifier 

## Append Data

In [4]:
df_2016 = pd.read_parquet('Data/01_Napel2016.parquet')
df_2016['Year'] = 2016
df_2022 = pd.read_parquet('Data/01_Napel2022.parquet')
df_2022['Year'] = 2022

In [5]:
df_all = pd.concat([df_2016, df_2022], axis = 0)

In [6]:
df_all.shape

(11568, 352)

In [7]:
df_all = df_all.dropna(axis=1, how="any")

In [8]:
df_all.shape

(11568, 277)

In [9]:
df_all = df_all.set_index(['PSU', 'HHLD'])

In [10]:
df_all.columns

Index(['EcoBelt', 'Prov', 'Rural_Dummy', 'Respon_Female', 'Respon_Age',
       'LivingYear', 'Edu_UnderSLC', 'Edu_Certificate', 'Edu_Bachelor',
       'Edu_Master',
       ...
       'SoilWaterConservationPast25', 'VisitClimateOfficePast25',
       'FoodConsumptionHabitPast25', 'OfffarmActiPast25',
       'NonFarmEmployPast25', 'FamilyMigrationPast25', 'RiskReductionPast25',
       'RoadImprovementPast25', 'CommunityPartipationPast25', 'Year'],
      dtype='object', length=275)

In [11]:
df_all["EcoBelt"] = df_all["EcoBelt"].str.replace('Tarai', 'Terai')

In [12]:
eco_dummies = pd.get_dummies(df_all["EcoBelt"], prefix="EcoBelt").astype(int)
df_all = pd.concat([df_all, eco_dummies], axis=1)

In [13]:
prov_dummies = pd.get_dummies(df_all["Prov"], prefix="Prov").astype(int)
df_all = pd.concat([df_all, prov_dummies], axis=1)

In [14]:
df_all

EcoBelt          Prov  Rural_Dummy  Respon_Female  Respon_Age  \
PSU   HHLD                                                                   
101.0 1.0   Mountain         Koshi            1            1.0        49.0   
      2.0   Mountain         Koshi            1            0.0        55.0   
      3.0   Mountain         Koshi            1            1.0        46.0   
      4.0   Mountain         Koshi            1            0.0        78.0   
      5.0   Mountain         Koshi            1            0.0        70.0   
...              ...           ...          ...            ...         ...   
826.0 16.0     Terai  Sudurpaschim            0            0.0        47.0   
      17.0     Terai  Sudurpaschim            0            0.0        60.0   
      18.0     Terai  Sudurpaschim            0            1.0        47.0   
      19.0     Terai  Sudurpaschim            0            0.0        55.0   
      20.0     Terai  Sudurpaschim            0            0.0        45.0   

            LivingYear  Edu_UnderSLC  Edu_Certificate  Edu_Bachelor  \
PSU   HHLD                                                            
101.0 1.0         49.0             0                0             0   
      2.0         55.0             1                0             0   
      3.0         25.0             0                0             0   
      4.0         78.0             0                0             0   
      5.0         70.0             1                0             0   
...                ...           ...              ...           ...   
826.0 16.0        40.0             1                0             0   
      17.0        35.0             0                0             0   
      18.0        32.0             1                0             0   
      19.0        55.0             0                0             0   
      20.0        25.0             1                0             0   

            Edu_Master  ...  EcoBelt_Hill  EcoBelt_Mountain  EcoBelt_Terai  \
PSU   HHLD              ...                                                  
101.0 1.0            0  ...             0                 1              0   
      2.0            0  ...             0                 1              0   
      3.0            0  ...             0                 1              0   
      4.0            0  ...             0                 1              0   
      5.0            0  ...             0                 1              0   
...                ...  ...           ...               ...            ...   
826.0 16.0           0  ...             0                 0              1   
      17.0           0  ...             0                 0              1   
      18.0           0  ...             0                 0              1   
      19.0           0  ...             0                 0              1   
      20.0           0  ...             0                 0              1   

            Prov_Bagmati  Prov_Gandaki  Prov_Karnali  Prov_Koshi  \
PSU   HHLD                                                         
101.0 1.0              0             0             0           1   
      2.0              0             0             0           1   
      3.0              0             0             0           1   
      4.0              0             0             0           1   
      5.0              0             0             0           1   
...                  ...           ...           ...         ...   
826.0 16.0             0             0             0           0   
      17.0             0             0             0           0   
      18.0             0             0             0           0   
      19.0             0             0             0           0   
      20.0             0             0             0           0   

            Prov_Lumbini  Prov_Madhesh  Prov_Sudurpaschim  
PSU   HHLD                                                 
101.0 1.0              0             0                

### Revise Income Resources

In [15]:
df_all['IncomeResAgri_dummy'] = (df_all[['IncomeS1', 'IncomeS2', 'IncomeS3']] == 1).any(axis=1).astype(int)
df_all['IncomeResWage_dummy'] = (df_all[['IncomeS1', 'IncomeS2', 'IncomeS3']] == 2).any(axis=1).astype(int)
df_all['IncomeResNonAgriBusi_dummy'] = (df_all[['IncomeS1', 'IncomeS2', 'IncomeS3']] == 3).any(axis=1).astype(int)
df_all['IncomeResRemit_dummy'] = (df_all[['IncomeS1', 'IncomeS2', 'IncomeS3']] == 4).any(axis=1).astype(int)
df_all['IncomeResOthers_dummy'] = (df_all[['IncomeS1', 'IncomeS2', 'IncomeS3']] == 5).any(axis=1).astype(int)

In [16]:
df_all['ResidenceOwn_dummy'] = (df_all['Own_Resid'] == 1).astype(int)
df_all['ResidenceRent_dummy'] = (df_all['Own_Resid'] == 2).astype(int)
df_all['ResidenceInstitu_dummy'] = (df_all['Own_Resid'] == 3).astype(int)
df_all['ResidenceOthers_dummy'] = (df_all['Own_Resid'] == 4).astype(int)

In [17]:
df_all['ResidInfraPerman_dummy'] = (df_all['Resid_Type'] == 1).astype(int)
df_all['ResidInfraSemi_dummy'] = (df_all['Resid_Type'] == 2).astype(int)
df_all['ResidInfraKachchi_dummy'] = (df_all['Resid_Type'] == 3).astype(int)
df_all['ResidInfraOthers_dummy'] = (df_all['Resid_Type'] == 4).astype(int)

## Checking "HeardClimate_Dummy"

### Dataset building

In [18]:
for here in range(0, df_all.shape[1], 40):
    print(df_all.columns[0+here:40+here])

Index(['EcoBelt', 'Prov', 'Rural_Dummy', 'Respon_Female', 'Respon_Age',
       'LivingYear', 'Edu_UnderSLC', 'Edu_Certificate', 'Edu_Bachelor',
       'Edu_Master', 'Edu_PhD', 'Edu_Literal', 'Edu_Illiterate', 'Edu_year',
       'Female_Ratio', 'U18_Ratio', 'A65_Ratio', 'Edu12_Ratio',
       'Literal_Ratio', 'Household_memberNum', 'Own_Resid', 'Resid_Type',
       'WaterS1', 'WaterS2', 'WaterS3', 'CookFuelS1', 'CookFuelS2',
       'CookFuelS3', 'LightEnergy', 'Toilet', 'IncomeS1', 'IncomeS2',
       'IncomeS3', 'Remittance_dummy', 'Have_AgriLand', 'Radio_dummy',
       'TV_dummy', 'PC_dummy', 'Net_dummy', 'Phone_dummy'],
      dtype='object')
Index(['Mobile_dummy', 'Motorbike_dummy', 'Car_dummy', 'Bike_dummy',
       'OtherVehi_dummy', 'Refrige_dummy', 'HouseHead_AgriExpYear',
       'SavingMembership', 'RegularSaving', 'OrgMembership', 'AgriSupport',
       'Dist_Road', 'Dist_HealthCenter', 'Dist_SecondarySchool', 'Dist_Market',
       'Dist_AgriSupport', 'FramMechan', 'CropIncome', 'L

In [19]:
df_inuse = df_all[['HeardClimate_Dummy', 
                   'Respon_Female', 'Respon_Age', 'LivingYear',  'Edu_Literal', 'Edu_Illiterate', 'Edu_year', # S01
                   'Female_Ratio', 'U18_Ratio', 'A65_Ratio', 'Edu12_Ratio', 'Literal_Ratio', # S02-1
                   'EcoBelt_Hill', 'EcoBelt_Mountain', 'EcoBelt_Terai', 
                   'Prov_Bagmati', 'Prov_Koshi', 'Prov_Lumbini', 'Prov_Madhesh', 'Prov_Sudurpaschim',
                   'Prov_Gandaki', 'Prov_Karnali', # location     
                   'ResidenceOwn_dummy', 'ResidenceRent_dummy', 'ResidenceInstitu_dummy', 'ResidenceOthers_dummy',
                   'ResidInfraPerman_dummy', 'ResidInfraSemi_dummy', 'ResidInfraKachchi_dummy', 'ResidInfraOthers_dummy', # house
                    'Remittance_dummy', 
                   'Have_AgriLand', 'HouseHead_AgriExpYear',
                   'Radio_dummy', 'TV_dummy', 'PC_dummy', 'Net_dummy', 'Phone_dummy',
                   'Mobile_dummy', 'Motorbike_dummy', 'Car_dummy', 'Bike_dummy', 'OtherVehi_dummy', 'Refrige_dummy',
                   'SavingMembership', 'RegularSaving', 'OrgMembership', 'AgriSupport', 
                   'Dist_Road', 'Dist_HealthCenter', 'Dist_SecondarySchool', 'Dist_Market', 'Dist_AgriSupport', 
                   'FramMechan',
                    'IncomeResAgri_dummy', 'IncomeResWage_dummy', 'IncomeResNonAgriBusi_dummy', 'IncomeResRemit_dummy',
                   'IncomeResOthers_dummy', 
                   'CropIncome', 'LivestockIncome', 'NonAgriIncome', 'BusiIncome', 'TotalIncome',
                   'Year']]

In [20]:
variname_readable = {'HeardClimate_Dummy':'Heard about Climate Change Dummy', 'Respon_Female':'Female Dummy', 
                     'Respon_Age':'Age', 'LivingYear':'Years Living in Community', 'Edu_UnderSLC':'Education under Secondary Certificate Dummy', 
                     'Edu_Certificate':'Education with Secondary Certificate Dummy', 'Edu_Bachelor':'Education with Bachelor Dummy', 
                     'Edu_Master':'Education with Master Dummy',  'Edu_PhD':'Education with PhD Dummy', 
                     'Edu_Literal':'Literate Education Dummy',  'Edu_Illiterate':'Illiterate Dummy', 'Edu_year':'Education Year',
                     'Female_Ratio':'Female Ratio in Household', 'U18_Ratio':'Member Under 18 Ratio', 'A65_Ratio':'Seniors Ratio',
                     'Edu12_Ratio':"Member with 12-Year Education or above Ratio", "Literal_Ratio": "Literate Member Ratio",
                     'EcoBelt_Hill': "EcoBelt Hill Dummy", 'EcoBelt_Mountain': "EcoBelt Mountain Dummy", 'EcoBelt_Terai': "EcoBelt Terai Dummy",
                     'Prov_Bagmati': "Province Bagmati Dummy", 'Prov_Koshi': "Province Koshi Dummy", 'Prov_Lumbini': "Province Lumbibi Dummy",
                     'Prov_Madhesh': "Province Madhesh Dummy", 'Prov_Sudurpaschim': "Province Sudurpaschim Dummy", 'Prov_Gandaki': "Province Gandaki Dummy",
                     'Prov_Karnali': "Province Karnali Dummy",
                     'ResidenceOwn_dummy': "Owned Residence Ownership Dummy", 'ResidenceRent_dummy': 'Rented Residence Ownership Dummy', 
                     'ResidenceInstitu_dummy': 'Institutional Residence Ownership Dummy', 'ResidenceOthers_dummy': 'Other-type Residence Ownership Dummy',
                     'ResidInfraPerman_dummy': 'Permanent Residence Dummy', 'ResidInfraSemi_dummy': 'Semi-Permanent Residence Dummy', 
                     'ResidInfraKachchi_dummy': "Kachchi Residence Dummy", 'ResidInfraOthers_dummy': 'Other Residence Infrastructure Dummy', # house
                     'Remittance_dummy' : "Have Remittance",
                     'Have_AgriLand': "Having Agricultural Land Dummy", 'HouseHead_AgriExpYear': "Household Head Agricultural Experience",
                     'Radio_dummy': "Having Radio Dummy", 'TV_dummy': "Having TV Dummy", 'PC_dummy': "Having Computer Dummy", 'Net_dummy': "Having Internet Dummy",
                     'Phone_dummy': "Having Telephone Dummy", 'Mobile_dummy': "Having Mobile Dummy", 'Motorbike_dummy': "Having Motorbike Dummy", 
                     'Car_dummy': "Having Car Dummy", 'Bike_dummy': "Having Bike Dummy", 'OtherVehi_dummy': "Having Other Vehicle Dummy", 
                     'Refrige_dummy':"Having Refrigator Dummy",
                     'SavingMembership': "Saving Membership Dummy", 'RegularSaving': 'Having Regular Saving Dummy', 
                     'OrgMembership': 'Having Organization Membership Dummy', 'AgriSupport':'Agricultural Supporting Dummy', 
                     'Dist_Road': 'Distance to Motorable Road', 'Dist_HealthCenter': "Distance to Health Center", 
                     'Dist_SecondarySchool': "Distance to Secondary School", 'Dist_Market':"Distance to Market", 'Dist_AgriSupport': 'Distance to Agricultural Center', 
                     'FramMechan':'Farm Mechanization Dummy',
                     'IncomeResAgri_dummy': "Agricultural Income Source Dummy", 'IncomeResWage_dummy': "Wage Income Source Dummy",
                     'IncomeResNonAgriBusi_dummy': "Non-Agricultural Business Income Source Dummy", 'IncomeResRemit_dummy': "Remittance Income Dummy",
                     'IncomeResOthers_dummy': "Others Income Source Dummy", 
                     'CropIncome': "Crop Income", 'LivestockIncome': "Livestock Income", 'NonAgriIncome': "Non-agricultural Income", 
                     'BusiIncome': "Business Income", 'TotalIncome': "Total Income",
                     'Year': "Survey Year"                    
                    }

In [21]:
data_summary = df_inuse.describe().T

In [22]:
data_summary.index = data_summary.index.map(variname_readable)

In [23]:
data_summary.to_excel("MLD01e_Results/Table01_DataSummaryForKnowledge_v1.xlsx")

In [23]:
df_inuse.columns = df_inuse.columns.map(variname_readable)

In [24]:
data_summary

,count,mean,std,min,25%,50%,75%,max
Heard about Climate Change Dummy,11568.0,0.421508,4.938219e-01,0.0,0.0,0.0,1.0,1.0
Female Dummy,11568.0,0.285183,4.515212e-01,0.0,0.0,0.0,1.0,1.0
Age,11568.0,58.681189,1.027651e+01,45.0,50.0,58.0,66.0,95.0
Years Living in Community,11568.0,50.082382,1.582942e+01,25.0,40.0,50.0,60.0,730.0
Literate Education Dummy,11568.0,0.169001,3.747687e-01,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...
Livestock Income,11568.0,15559.887275,9.565885e+04,0.0,0.0,0.0,15000.0,7200000.0
Non-agricultural Income,11568.0,79973.158368,2.710275e+05,0.0,0.0,0.0,50000.0,10000000.0
Business Income,11568.0,202175.389004,1.112664e+06,0.0,0.0,15000.0,240000.0,99000000.0
Total Income,11568.0,323942.349066,1.143592e+06,0.0,54000.0,170000.0,400000.0,99200000.0


In [25]:
cvres = pd.read_parquet('MLD01e_Results/MLD01e_C01_KnowledgeFactorInvestigation.parquet')

In [26]:
cvres.sort_values('rank_test_score').iloc[0,:]

mean_fit_time                                                      2.661557
std_fit_time                                                       0.133185
mean_score_time                                                    0.018475
std_score_time                                                     0.003636
param_subsample                                                         0.7
param_n_estimators                                                      500
param_min_child_weight                                                    5
param_max_depth                                                           8
param_learning_rate                                                    0.01
param_colsample_bytree                                                  0.7
params                    {'colsample_bytree': 0.7, 'learning_rate': 0.0...
split0_test_score                                                   0.75108
split1_test_score                                                  0.760588
split2_test_

In [27]:
params = cvres.sort_values('rank_test_score').iloc[0,10]

In [29]:
params

{'colsample_bytree': 0.7,
 'learning_rate': 0.01,
 'max_depth': 8,
 'min_child_weight': 5,
 'n_estimators': 500,
 'subsample': 0.7}

### Logistic

In [30]:
def Metrics(y_test, y_pred):
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, zero_division=0))
    print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
    print("F1-score :", f1_score(y_test, y_pred, zero_division=0))

    return [accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, zero_division=0),
    recall_score(y_test, y_pred, zero_division=0), f1_score(y_test, y_pred, zero_division=0)]

In [31]:
y = df_inuse['Heard about Climate Change Dummy'].astype(int)
X = df_inuse.drop(columns=['Heard about Climate Change Dummy'])

pos = y.sum()
neg = len(y) - pos
scale_pos_weight = (neg / pos) if pos > 0 else 1.0

In [48]:
acc_lines = []

random_seed = 42
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    pos = y_train.sum()
    neg = len(y_train) - pos
    scale_pos_weight = (neg / pos) if pos > 0 else 1.0

    class_weight = {0: 1, 1: scale_pos_weight}

    print(f'fold: {fold}, random_seed:{random_seed}')
    clf = LogisticRegression(class_weight=class_weight, n_jobs = -1, penalty = None)
    clf.fit(X_train, y_train)

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    metrics = Metrics(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred).flatten()
    line  = [fold, random_seed] + metrics + list(cm)

    acc_lines.append(line)

fold: 1, random_seed:42


/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

Accuracy: 0.6248919619706137
Precision: 0.5691906005221932
Recall   : 0.44763860369609854
F1-score : 0.5011494252873563
fold: 2, random_seed:42


/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

Accuracy: 0.6240276577355229
Precision: 0.5714285714285714
Recall   : 0.4271047227926078
F1-score : 0.4888366627497062
fold: 3, random_seed:42


...

Accuracy: 0.6145203111495247
Precision: 0.5552631578947368
Recall   : 0.4323770491803279
F1-score : 0.4861751152073733
fold: 9, random_seed:42


/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

Accuracy: 0.6219723183391004
Precision: 0.5694444444444444
Recall   : 0.4209445585215606
F1-score : 0.48406139315230223
fold: 10, random_seed:42


/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/lichao/opt/anaconda3/envs/ML/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linea

Accuracy: 0.6444636678200693
Precision: 0.5974358974358974
Recall   : 0.4784394250513347
F1-score : 0.5313568985176739


In [49]:
acc_df = pd.DataFrame(acc_lines)

In [50]:
acc_df.columns = ['fold', 'random_seed', 'acc', 'prec', 'recall', 'f1', 'TN', 'FP', 'FN', 'TP']

In [51]:
acc_df.to_excel('MLD01e_Results/Table05_AccuracyTable_Logistic_Knowledge_v1.xlsx')

In [52]:
acc_df[['acc', 'prec', 'recall', 'f1']].mean(axis=0)

acc       0.629669
prec      0.576208
recall    0.470650
f1        0.515982
dtype: float64

In [54]:
acc_df[['acc', 'prec', 'recall', 'f1']].std(axis=0)

acc       0.015777
prec      0.028830
recall    0.054298
f1        0.029172
dtype: float64

In [53]:
acc_df

,fold,random_seed,acc,prec,recall,f1,TN,FP,FN,TP
0,1,42,0.624892,0.569191,0.447639,0.501149,505,165,269,218
1,2,42,0.624028,0.571429,0.427105,0.488837,514,156,279,208
2,3,42,0.624892,0.574176,0.428279,0.490610,514,155,279,209
3,4,42,0.611063,0.535985,0.579918,0.557087,424,245,205,283
4,5,42,0.665514,0.644699,0.461066,0.537634,545,124,263,225
5,6,42,0.630942,0.564211,0.549180,0.556594,462,207,220,268
6,7,42,0.634399,0.580247,0.481557,0.526316,499,170,253,235
7,8,42,0.614520,0.555263,0.432377,0.486175,500,169,277,211
8,9,42,0.621972,0.569444,0.420945,0.484061,514,155,282,205
9,10,42,0.644464,0.597436,0.478439,0.531357,512,157,254,233


### XGBoost

In [55]:
cvres = pd.read_parquet('MLD01e_Results/MLD01e_C01_KnowledgeFactorInvestigation.parquet')

In [56]:
params = cvres.sort_values('rank_test_score').iloc[0,10]

In [57]:
params

{'colsample_bytree': 0.7,
 'learning_rate': 0.01,
 'max_depth': 8,
 'min_child_weight': 5,
 'n_estimators': 500,
 'subsample': 0.7}

In [58]:
acc_lines = []

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=random_seed)
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    pos = y_train.sum()
    neg = len(y_train) - pos
    scale_pos_weight = (neg / pos) if pos > 0 else 1.0

    print(f'fold: {fold}, random_seed:{random_seed}')
    clf = XGBClassifier(objective="binary:logistic",
                        eval_metric="logloss", tree_method="hist", 
                        random_state=random_seed, 
                        scale_pos_weight=scale_pos_weight, device = 'cuda',
                        **params
    )

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)


    metrics = Metrics(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred).flatten()
    line  = [fold, random_seed] + metrics + list(cm)

    acc_lines.append(line)

fold: 1, random_seed:42
Accuracy: 0.7467588591184097
Precision: 0.6963562753036437
Recall   : 0.7063655030800822
F1-score : 0.7013251783893986
fold: 2, random_seed:42
Accuracy: 0.766637856525497
Precision: 0.7237113402061855
Recall   : 0.7207392197125256
F1-score : 0.7222222222222222
fold: 3, random_seed:42
Accuracy: 0.7631806395851339
Precision: 0.717479674796748
Recall   : 0.7233606557377049
F1-score : 0.7204081632653061
fold: 4, random_seed:42
Accuracy: 0.7536732929991357
Precision: 0.704225352112676
Recall   : 0.7172131147540983
F1-score : 0.7106598984771574
fold: 5, random_seed:42
Accuracy: 0.7683664649956785
Precision: 0.7217741935483871
Recall   : 0.7336065573770492
F1-score : 0.7276422764227642
fold: 6, random_seed:42
Accuracy: 0.7605877268798618
Precision: 0.7268817204301076
Recall   : 0.6926229508196722
F1-score : 0.7093389296956978
fold: 7, random_seed:42
Accuracy: 0.7778738115816768
Precision: 0.7431578947368421
Recall   : 0.7233606557377049
F1-score : 0.7331256490134995
fo

In [59]:
acc_df_xgb = pd.DataFrame(acc_lines)

In [60]:
acc_df_xgb.columns = ['fold', 'random_seed', 'acc', 'prec', 'recall', 'f1', 'TN', 'FP', 'FN', 'TP']

In [61]:
acc_df_xgb[['acc', 'prec', 'recall', 'f1']].mean(axis=0)

acc       0.760026
prec      0.714491
recall    0.718214
f1        0.716197
dtype: float64

In [62]:
acc_df_xgb[['acc', 'prec', 'recall', 'f1']].std(axis=0)

acc       0.010266
prec      0.016918
recall    0.012329
f1        0.010099
dtype: float64

In [63]:
acc_df_xgb

,fold,random_seed,acc,prec,recall,f1,TN,FP,FN,TP
0,1,42,0.746759,0.696356,0.706366,0.701325,520,150,143,344
1,2,42,0.766638,0.723711,0.720739,0.722222,536,134,136,351
2,3,42,0.763181,0.717480,0.723361,0.720408,530,139,135,353
3,4,42,0.753673,0.704225,0.717213,0.710660,522,147,138,350
4,5,42,0.768366,0.721774,0.733607,0.727642,531,138,130,358
5,6,42,0.760588,0.726882,0.692623,0.709339,542,127,150,338
6,7,42,0.777874,0.743158,0.723361,0.733126,547,122,135,353
7,8,42,0.758859,0.716356,0.709016,0.712667,532,137,142,346
8,9,42,0.761246,0.712274,0.726899,0.719512,526,143,133,354
9,10,42,0.743080,0.682692,0.728953,0.705065,504,165,132,355
